In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, GPT2Model

##################################################################
# 1) Configuration
##################################################################
opts = {
    "model_nm": "gpt2",
    "wiki_data": "wikitext",
    "wiki_cfg": "wikitext-2-raw-v1",
    "wiki_split": "train",
    "fin_data": "financial_phrasebank",
    "fin_cfg": "sentences_50agree",
    "fin_split": "train",
    "max_items": 30000,
    "batch_sz": 64,
    "drift_std": 4.0,
    "win_sz": 50,
    "var_thresh": 0.0001,
    "out_dir": "results_jupyter_gpt2_fixed",
}


##################################################################
# 2) Helpers
##################################################################
def feed_batches(data, sz=32):
    for start in range(0, len(data), sz):
        yield data[start : start + sz]


def extract_cls_embeddings(model, tokenizer, texts, device):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tok_out = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=512
    )
    ids = tok_out["input_ids"].to(device)
    msk = tok_out["attention_mask"].to(device)
    with torch.no_grad():
        res = model(ids, attention_mask=msk)
        first_tok = res.last_hidden_state[:, 0, :]
    return first_tok.mean(dim=0).cpu().numpy()


def cluster_outliers(points, gap=128, remove_solo=False, min_clust=2):
    if not points:
        return []
    points = sorted(points)
    blocks = []
    temp = [points[0]]
    for val in points[1:]:
        if val - temp[-1] <= gap:
            temp.append(val)
        else:
            blocks.append(temp)
            temp = [val]
    blocks.append(temp)
    if remove_solo:
        blocks = [b for b in blocks if len(b) >= min_clust]
    return blocks


def pick_cluster_index(clust, mode="first"):
    out = []
    for c in clust:
        if mode == "first":
            out.append(c[0])
        elif mode == "last":
            out.append(c[-1])
        elif mode == "mean":
            out.append(int(sum(c) / len(c)))
        elif mode == "median":
            c_sort = sorted(c)
            m = len(c_sort) // 2
            if len(c_sort) % 2 == 1:
                out.append(c_sort[m])
            else:
                out.append(int((c_sort[m - 1] + c_sort[m]) / 2))
    return out


##################################################################
# 3) Detector
##################################################################
class ShiftDetector:
    def __init__(self, model, tokenizer, device, bgen, options):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.bgen = bgen
        self.options = options
        self.baseline = None
        self.baselines = []
        self.alerts = []
        self.scores = []

    def init_baseline(self, docs):
        pile = []
        chunk = docs[: self.options["max_items"]]
        for pack in self.bgen(chunk, self.options["batch_sz"]):
            part = extract_cls_embeddings(self.model, self.tokenizer, pack, self.device)
            pile.append(part[np.newaxis, :])
        comb = np.concatenate(pile, axis=0)
        self.baseline = np.mean(comb, axis=0)
        self.baselines.append(self.baseline)

    def scan(self, docs):
        for i, pack in enumerate(tqdm(self.bgen(docs, self.options["batch_sz"]))):
            local_pile = []
            emb = extract_cls_embeddings(self.model, self.tokenizer, pack, self.device)
            local_pile.append(emb[np.newaxis, :])
            batch_arr = np.concatenate(local_pile, axis=0)
            rep = np.mean(batch_arr, axis=0, keepdims=True)
            sim = cosine_similarity(rep, [self.baseline])[0][0]
            self.scores.append(sim)
            if len(self.scores) >= self.options["win_sz"]:
                threshold = self._calc_threshold()
                variance = self._calc_variance()
                if sim < threshold or (
                    variance is not None and variance < self.options["var_thresh"]
                ):
                    self.alerts.append(i * self.options["batch_sz"])
            self._update_baseline(batch_arr)

    def _calc_threshold(self):
        sub = self.scores[-self.options["win_sz"] :]
        mu = np.mean(sub)
        sd = np.std(sub)
        return mu - self.options["drift_std"] * sd

    def _calc_variance(self):
        if len(self.scores) < self.options["win_sz"]:
            return None
        sub = self.scores[-self.options["win_sz"] :]
        return np.var(sub)

    def _update_baseline(self, arr):
        delta = arr - self.baseline
        dist = np.linalg.norm(delta, axis=1)
        wts = np.exp(-dist / 2.0)
        wts_sum = np.sum(wts[:, None] * delta, axis=0)
        self.baseline += wts_sum / np.sum(wts)
        self.baselines.append(self.baseline)


##################################################################
# 4) Main
##################################################################
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

os.makedirs(opts["out_dir"], exist_ok=True)
tok = AutoTokenizer.from_pretrained(opts["model_nm"])
mdl = GPT2Model.from_pretrained(opts["model_nm"])
mdl.to(device)
mdl.eval()

wiki_ds = load_dataset(opts["wiki_data"], opts["wiki_cfg"], split=opts["wiki_split"])
stuff = wiki_ds["text"]
if opts["max_items"] > 0 and opts["max_items"] < len(stuff):
    stuff = stuff[: opts["max_items"]]

fin_ds = load_dataset(opts["fin_data"], opts["fin_cfg"], split=opts["fin_split"])
fin_txt = fin_ds["sentence"]
random.shuffle(fin_txt)
kaggle_set = fin_txt[:1000]

lng = len(stuff)
s1, e1 = lng // 3, lng // 2
s2, e2 = 2 * lng // 3, 5 * lng // 6
idx_f = 0
for i in range(s1, e1):
    stuff[i] = fin_txt[idx_f % len(fin_txt)]
    idx_f += 1
idx_k = 0
for i in range(s2, e2):
    stuff[i] = kaggle_set[idx_k % len(kaggle_set)]
    idx_k += 1

detect = ShiftDetector(mdl, tok, device, feed_batches, opts)
detect.init_baseline(stuff)
detect.scan(stuff)

grp = cluster_outliers(points=detect.alerts, gap=128, remove_solo=True, min_clust=4)
rep_idx = pick_cluster_index(grp, mode="first")

plt.figure(figsize=(12, 8))
plt.plot(detect.scores, label="Cosine Similarity", linewidth=2)
plt.axvspan(s1 / opts["batch_sz"], e1 / opts["batch_sz"], color="orange", alpha=0.2)
plt.axvspan(s2 / opts["batch_sz"], e2 / opts["batch_sz"], color="blue", alpha=0.2)
for r in rep_idx:
    bx = r / opts["batch_sz"]
    if 0 <= bx < len(detect.scores):
        plt.axvline(bx, color="red", linestyle="--")
        plt.scatter(bx, detect.scores[int(bx)], color="red", zorder=5)
plt.title("Cosine Similarity with Drift Regions", fontsize=16)
plt.xlabel("Batch Index", fontsize=14)
plt.ylabel("Cosine Similarity", fontsize=14)
plt.legend(loc="best", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
save_plot = os.path.join(opts["out_dir"], "drift_plot_gpt2_fixed.png")
plt.savefig(save_plot)
plt.show()

4